Dimensioni (condivise tra tutte le fact table):
- dim_cliente   - da silver_clienti_assets (fonte di verita')
- dim_articolo  - da silver_articoli_assets (fonte di verita')
- dim_data      - calendario standard

Fact table (tenute separate, vedi discussione: storico vs nuovi documenti sono semanticamente diversi - lo storico e' gia' transato/confermato, i nuovi ordini da PDF sono output di una pipeline AI con stati intermedi):
- fact_ordini_storici  - da silver_ordini_storici + silver_ordini_righe_storiche
- fact_ordini_pdf      - da silver_ordini + silver_ordini_righe
- fact_quotazioni_pdf  - da silver_quotazioni + silver_quotazioni_righe

REGOLA SUL PREZZO: il prezzo usato per calcolare fatturato/valore e' SEMPRE quello di listino preso da dim_articolo (fonte Assets), mai quello dichiarato nel documento. Il prezzo dichiarato resta come colonna informativa separata, utile solo per analizzare gli scostamenti rispetto
al listino - non entra mai nei totali di fatturato.

In [4]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

StatementMeta(, 3fb3777a-e3dc-4673-b792-2ab7c2798807, 8, Finished, Available, Finished, False)

In [5]:
# DIM_CLIENTE - fonte di verita': Assets

df_clienti_assets = spark.sql("SELECT * FROM silver_clienti_assets")
 
dim_cliente = df_clienti_assets.select(
    F.col("id").alias("id_cliente"),
    F.col("ragioneSociale").alias("ragione_sociale"),
    F.col("partitaIva").alias("partita_iva"),
    F.col("indirizzo"),
    F.col("cap"),
    F.col("citta"),
    F.col("provincia"),
    F.col("email"),
    F.col("telefono"),
)
 
dim_cliente.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("dim_cliente")
print(f"dim_cliente: {dim_cliente.count()} righe")

StatementMeta(, 3fb3777a-e3dc-4673-b792-2ab7c2798807, 9, Finished, Available, Finished, False)

dim_cliente: 20 righe


In [6]:
# DIM_ARTICOLO - fonte di verita': Assets (prezzo di listino ufficiale)

df_articoli_assets = spark.sql("SELECT * FROM silver_articoli_assets")
 
dim_articolo = df_articoli_assets.select(
    F.col("codice").alias("codice_articolo"),
    F.col("descrizione"),
    F.col("categoria"),
    F.col("unitaMisura").alias("unita_misura"),
    F.col("prezzoListino").alias("prezzo_listino"),
)
 
dim_articolo.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("dim_articolo")
print(f"dim_articolo: {dim_articolo.count()} righe")

StatementMeta(, 3fb3777a-e3dc-4673-b792-2ab7c2798807, 10, Finished, Available, Finished, False)

dim_articolo: 40 righe


In [7]:
# DIM_DATA - calendario standard, costruito sull'intervallo di date presenti
# nei dati (storico + nuovi), con un margine di sicurezza
 
date_min_max = spark.sql("""
    SELECT
        MIN(d) AS data_min,
        MAX(d) AS data_max
    FROM (
        SELECT data_ordine AS d FROM silver_ordini_storici WHERE data_ordine IS NOT NULL
        UNION ALL
        SELECT data_ordine AS d FROM silver_ordini WHERE data_ordine IS NOT NULL
        UNION ALL
        SELECT data_richiesta AS d FROM silver_quotazioni WHERE data_richiesta IS NOT NULL
    )
""").collect()[0]
 
data_min = date_min_max["data_min"]
data_max = date_min_max["data_max"]
 
if data_min is None or data_max is None:
    # fallback se non ci sono ancora date valide nei dati
    data_min, data_max = "2024-01-01", "2027-12-31"
 
dim_data = (
    spark.sql(f"SELECT explode(sequence(to_date('{data_min}'), to_date('{data_max}'), interval 1 day)) AS data")
    .withColumn("id_data", F.date_format(F.col("data"), "yyyyMMdd").cast("int"))
    .withColumn("anno", F.year("data"))
    .withColumn("mese", F.month("data"))
    .withColumn("nome_mese", F.date_format("data", "MMMM"))
    .withColumn("trimestre", F.quarter("data"))
    .withColumn("giorno_settimana", F.date_format("data", "EEEE"))
    .withColumn("settimana_anno", F.weekofyear("data"))
    .select("id_data", "data", "anno", "mese", "nome_mese", "trimestre", "giorno_settimana", "settimana_anno")
)
 
dim_data.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("dim_data")
print(f"dim_data: {dim_data.count()} righe (da {data_min} a {data_max})")

StatementMeta(, 3fb3777a-e3dc-4673-b792-2ab7c2798807, 11, Finished, Available, Finished, False)

dim_data: 502 righe (da 2025-01-13 a 2026-05-29)


In [8]:
# FACT_ORDINI_STORICI - join righe storiche + testata storica + dim_articolo
# per il prezzo di listino ufficiale (mai il prezzo dichiarato per i totali)

df_ordini_storici = spark.sql("SELECT * FROM silver_ordini_storici")
df_righe_storiche = spark.sql("SELECT * FROM silver_ordini_righe_storiche")

fact_ordini_storici = (
    df_righe_storiche.alias("r")
    .join(df_ordini_storici.alias("o"), F.col("r.id_ordine_storico") == F.col("o.id_ordine_storico"), "left")
    .join(dim_articolo.alias("a"), F.col("r.codice_articolo_match") == F.col("a.codice_articolo"), "left")
    .withColumn("id_data", F.date_format(F.col("o.data_ordine"), "yyyyMMdd").cast("int"))
    .select(
        F.col("o.id_ordine_storico").alias("id_ordine"),
        F.col("o.id_cliente"),
        F.col("id_data"),
        F.col("r.codice_articolo_match").alias("codice_articolo"),
        F.col("r.quantita"),
        F.col("a.prezzo_listino"),  # SEMPRE da Assets/dim_articolo
        F.col("r.prezzo_dichiarato"),  # colonna informativa, non usata nei totali
        (F.col("r.quantita") * F.col("a.prezzo_listino")).alias("valore_riga"),
        F.col("r.anomalia_integrita"),
        # Colonna unificata con fact_ordini_pdf: permette al dataflow di
        # filtrare le righe non ancora validate senza dover distinguere
        # tra le due fonti. Storico = gia' transato, quindi richiede
        # revisione solo se e' emersa un'anomalia di integrita'.
        F.when(F.col("r.anomalia_integrita").isNotNull(), F.lit(True))
         .otherwise(F.lit(False)).alias("richiede_revisione"),
        F.lit("storico_assets").alias("fonte_dato"),
    )
)

fact_ordini_storici.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("fact_ordini_storici")
print(f"\nfact_ordini_storici: {fact_ordini_storici.count()} righe")


StatementMeta(, 3fb3777a-e3dc-4673-b792-2ab7c2798807, 12, Finished, Available, Finished, False)


fact_ordini_storici: 400 righe


In [9]:
# FACT_ORDINI_PDF - join righe ordini estratti da PDF + testata + dim_articolo

df_ordini_pdf = spark.sql("SELECT * FROM silver_ordini")
df_righe_pdf = spark.sql("SELECT * FROM silver_ordini_righe")

fact_ordini_pdf = (
    df_righe_pdf.alias("r")
    .join(df_ordini_pdf.alias("o"), F.col("r.pdf_nome") == F.col("o.pdf_nome"), "left")
    .join(dim_articolo.alias("a"), F.col("r.codice_articolo_match") == F.col("a.codice_articolo"), "left")
    .withColumn("id_data", F.date_format(F.col("o.data_ordine"), "yyyyMMdd").cast("int"))
    .select(
        F.col("o.pdf_nome").alias("id_ordine"),
        F.col("o.id_cliente"),
        F.col("id_data"),
        F.col("r.codice_articolo_match").alias("codice_articolo"),
        F.col("r.quantita"),
        F.col("a.prezzo_listino"),  # SEMPRE da Assets/dim_articolo
        F.col("r.prezzo_dichiarato"),  # colonna informativa, non usata nei totali
        (F.col("r.quantita") * F.col("a.prezzo_listino")).alias("valore_riga"),
        F.col("r.confidenza_match"),
        F.col("r.note_anomalia"),
        F.col("o.intervento_umano_necessario"),
        # Colonna unificata con fact_ordini_storici (vedi sopra): stesso
        # nome/semantica, cosi' il dataflow puo' fare un filtro unico
        # su entrambe le fonti dopo l'append, invece di perdere questa
        # informazione nel Table.SelectColumns come accadeva finora.
        F.col("o.intervento_umano_necessario").alias("richiede_revisione"),
        F.lit("pdf_estratto").alias("fonte_dato"),
    )
)

fact_ordini_pdf.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("fact_ordini_pdf")
print(f"fact_ordini_pdf: {fact_ordini_pdf.count()} righe")


StatementMeta(, 3fb3777a-e3dc-4673-b792-2ab7c2798807, 13, Finished, Available, Finished, False)

fact_ordini_pdf: 166 righe


In [ ]:
# CODA_REVISIONE_ORDINI - vista aggregata A LIVELLO ORDINE (non riga) degli
# ordini che richiedono intervento umano. Un operatore lavora su documenti
# interi, non su singole righe - se un ordine ha 8 righe e una sola ha
# un'anomalia, l'operatore deve comunque riaprire l'intero documento.
# Aggregare qui evita di presentare righe duplicate per lo stesso ordine.
#
# Include il nome cliente ESTRATTO dal documento anche quando non e' stato
# risolto in anagrafica (id_cliente_risolto = null): senza questo,
# l'operatore vedrebbe solo "cliente sconosciuto" senza sapere COSA
# controllare per decidere se e' un cliente nuovo da censire o un errore
# nel documento.

# --- Ordini da PDF che richiedono revisione ---
df_ordini_pdf_full = spark.sql("SELECT * FROM silver_ordini")
righe_pdf_revisione = fact_ordini_pdf.filter(F.col("richiede_revisione") == True)

coda_pdf = (
    righe_pdf_revisione.alias("f")
    .join(df_ordini_pdf_full.alias("o"), F.col("f.id_ordine") == F.col("o.pdf_nome"), "left")
    .groupBy("f.id_ordine")
    .agg(
        F.first("o.cliente_ragione_sociale").alias("cliente_ragione_sociale_estratta"),
        F.first("o.cliente_partita_iva").alias("cliente_partita_iva_estratta"),
        F.first("f.id_cliente").alias("id_cliente_risolto"),
        F.first("o.riferimento_ordine").alias("riferimento_ordine"),
        F.first("o.data_ordine").alias("data_ordine"),
        F.count("*").alias("numero_righe_totali"),
        F.sum(F.when(F.col("f.note_anomalia").isNotNull(), 1).otherwise(0)).alias("righe_con_anomalia"),
        F.concat_ws(" | ", F.collect_set("f.note_anomalia")).alias("dettaglio_anomalie"),
        F.sum(F.coalesce(F.col("f.valore_riga"), F.lit(0.0))).alias("valore_stimato_totale"),
    )
    .withColumn(
        "motivo_principale",
        F.when(F.col("id_cliente_risolto").isNull(), F.lit("Cliente non risolto in anagrafica"))
         .otherwise(F.lit("Anomalia su una o piu' righe"))
    )
    .withColumn("fonte_dato", F.lit("pdf_estratto"))
)

# --- Ordini storici che richiedono revisione (anomalie di integrita') ---
df_ordini_storici_full = spark.sql("SELECT * FROM silver_ordini_storici")
righe_storiche_revisione = fact_ordini_storici.filter(F.col("richiede_revisione") == True)

coda_storici = (
    righe_storiche_revisione.alias("f")
    .join(df_ordini_storici_full.alias("o"), F.col("f.id_ordine") == F.col("o.id_ordine_storico"), "left")
    .groupBy("f.id_ordine")
    .agg(
        F.first("o.cliente_ragione_sociale").alias("cliente_ragione_sociale_estratta"),
        F.first("o.cliente_partita_iva").alias("cliente_partita_iva_estratta"),
        F.first("f.id_cliente").alias("id_cliente_risolto"),
        F.first("o.riferimento_ordine").alias("riferimento_ordine"),
        F.first("o.data_ordine").cast("string").alias("data_ordine"),
        F.count("*").alias("numero_righe_totali"),
        F.sum(F.when(F.col("f.anomalia_integrita").isNotNull(), 1).otherwise(0)).alias("righe_con_anomalia"),
        F.concat_ws(" | ", F.collect_set("f.anomalia_integrita")).alias("dettaglio_anomalie"),
        F.sum(F.coalesce(F.col("f.valore_riga"), F.lit(0.0))).alias("valore_stimato_totale"),
    )
    .withColumn("motivo_principale", F.lit("Anomalia di integrita' nello storico"))
    .withColumn("fonte_dato", F.lit("storico_assets"))
)

coda_revisione_ordini = coda_pdf.unionByName(coda_storici)

coda_revisione_ordini.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("coda_revisione_ordini")
print(f"coda_revisione_ordini: {coda_revisione_ordini.count()} ordini da rivedere")

In [15]:
# FACT_QUOTAZIONI_PDF - join righe quotazione + testata + dim_articolo
# (qui il prezzo e' gia' sempre quello di listino per costruzione, vedi
# prepara_quotazione nel notebook 01: la quotazione calcola dal listino,
# quindi non c'e' un "prezzo dichiarato" da confrontare)
 
df_quotazioni_pdf = spark.sql("SELECT * FROM silver_quotazioni")
df_righe_quotazioni = spark.sql("SELECT * FROM silver_quotazioni_righe")
 
fact_quotazioni_pdf = (
    df_righe_quotazioni.alias("r")
    .join(df_quotazioni_pdf.alias("q"), F.col("r.pdf_nome") == F.col("q.pdf_nome"), "left")
    .withColumn("id_data", F.date_format(F.col("q.data_richiesta"), "yyyyMMdd").cast("int"))
    .select(
        F.col("q.pdf_nome").alias("id_quotazione"),
        F.col("q.id_cliente"),
        F.col("id_data"),
        F.col("r.codice_articolo_match").alias("codice_articolo"),
        F.col("r.quantita"),
        F.col("r.prezzo_unitario_listino").alias("prezzo_listino"),
        F.col("r.importo_riga").alias("valore_riga"),
        F.col("r.confidenza_match"),
        F.lit("pdf_estratto").alias("fonte_dato"),
    )
)
 
fact_quotazioni_pdf.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("fact_quotazioni_pdf")
print(f"fact_quotazioni_pdf: {fact_quotazioni_pdf.count()} righe")

StatementMeta(, 28e8d8b8-e432-4f70-b441-ee8384adc7d9, 17, Finished, Available, Finished, False)

fact_quotazioni_pdf: 6 righe


In [16]:
print("\nCompletato. Star schema Gold creato/aggiornato:")
for nome_tabella in [
    "dim_cliente", "dim_articolo", "dim_data",
    "fact_ordini_storici", "fact_ordini_pdf", "fact_quotazioni_pdf",
    "coda_revisione_ordini",
]:
    print(f"  - {nome_tabella}")

StatementMeta(, 28e8d8b8-e432-4f70-b441-ee8384adc7d9, 18, Finished, Available, Finished, False)


Completato. Star schema Gold creato/aggiornato:
  - dim_cliente
  - dim_articolo
  - dim_data
  - fact_ordini_storici
  - fact_ordini_pdf
  - fact_quotazioni_pdf
